# Hands-On Lab — EDA & Cleaning on 911 Calls
--- 
## Lab 3  — Full Data Preparation on 911.csv

Apply the full data preparation pipeline to the 911 Calls dataset. You will explore, clean, encode, scale, and split the data.
- Task 1: Run a full EDA: shape, dtypes, describe, null counts, and value_counts on dept and reason
- Task 2: Clean the dataset: fill missing zip codes, drop duplicate rows, fix timeStamp to datetime type
- Task 3: Engineer features: extract hour, day of week, and month from the timeStamp column
- Task 4: Apply One-Hot Encoding to the dept column using pd.get_dummies()
- Task 5: Scale the lat and lng columns using MinMaxScaler from scikit-learn
- Task 6: Perform a train/test split (80/20) — confirm shapes of X_train and X_test

**Dataset**: [Emergency - 911 Calls](https://www.kaggle.com/datasets/mchirico/montcoalert)

## Exploratory Data Analysis 

In [1]:
# Import the necessary libraries 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

In [2]:
# Load Dataset
df_calls = pd.read_csv('./911.csv')

df_calls.head()

,lat,lng,desc,zip,title,timeStamp,twp,addr,e
0,40.297876,-75.581294,REINDEER CT & DEAD END; NEW HANOVER; Station ...,19525.0,EMS: BACK PAINS/INJURY,2015-12-10 17:10:52,NEW HANOVER,REINDEER CT & DEAD END,1
1,40.258061,-75.264680,BRIAR PATH & WHITEMARSH LN; HATFIELD TOWNSHIP...,19446.0,EMS: DIABETIC EMERGENCY,2015-12-10 17:29:21,HATFIELD TOWNSHIP,BRIAR PATH & WHITEMARSH LN,1
2,40.121182,-75.351975,HAWS AVE; NORRISTOWN; 2015-12-10 @ 14:39:21-St...,19401.0,Fire: GAS-ODOR/LEAK,2015-12-10 14:39:21,NORRISTOWN,HAWS AVE,1
3,40.116153,-75.343513,AIRY ST & SWEDE ST; NORRISTOWN; Station 308A;...,19401.0,EMS: CARDIAC EMERGENCY,2015-12-10 16:47:36,NORRISTOWN,AIRY ST & SWEDE ST,1
4,40.251492,-75.603350,CHERRYWOOD CT & DEAD END; LOWER POTTSGROVE; S...,NaN,EMS: DIZZINESS,2015-12-10 16:56:52,LOWER POTTSGROVE,CHERRYWOOD CT & DEAD END,1


In [3]:
# Shape and Dtypes
print(f"Shape: {df_calls.shape}\n")
print(f"Data Types:\n{df_calls.dtypes}")

Shape: (663522, 9)

Data Types:
lat          float64
lng          float64
desc             str
zip          float64
title            str
timeStamp        str
twp              str
addr             str
e              int64
dtype: object


In [4]:
# Statistical summary
df_calls.describe()

,lat,lng,zip,e
count,663522.000000,663522.000000,583323.000000,663522.0
mean,40.158162,-75.300105,19236.055791,1.0
std,0.220641,1.672884,298.222637,0.0
min,0.000000,-119.698206,1104.000000,1.0
25%,40.100344,-75.392735,19038.000000,1.0
50%,40.143927,-75.305143,19401.000000,1.0
75%,40.229008,-75.211865,19446.000000,1.0
max,51.335390,87.854975,77316.000000,1.0


In [5]:
# NUll Counts
print(f"Null Counts:\n{df_calls.isnull().sum()}")

Null Counts:
lat              0
lng              0
desc             0
zip          80199
title            0
timeStamp        0
twp            293
addr             0
e                0
dtype: int64


Before we take value counts on Department and reason, we must first engineer these new features

In [6]:
# Split into two columns
df_calls[["dept", "reason"]] = df_calls["title"].str.split(':', n=1, expand=True)


In [7]:
# Display value counts of dept and reason

dept_counts = (
    df_calls['dept'].value_counts().reset_index())

dept_counts.columns = ['Department', 'Count']

reason_counts = (
    df_calls['reason'].value_counts().reset_index())
reason_counts.columns = ['Reason', 'Count']

print(f"Department Value Counts:\n{dept_counts}")
print()
print(f"Reason Value Counts:\n{reason_counts}")

Department Value Counts:
  Department   Count
0        EMS  332692
1    Traffic  230208
2       Fire  100622

Reason Value Counts:
                  Reason   Count
0     VEHICLE ACCIDENT -  148372
1     DISABLED VEHICLE -   47909
2             FIRE ALARM   38452
3       VEHICLE ACCIDENT   36377
4            FALL VICTIM   34683
..                   ...     ...
90      ROAD OBSTRUCTION       2
91             HIT + RUN       1
92      ANIMAL COMPLAINT       1
93   PRISONER IN CUSTODY       1
94           FOOT PATROL       1

[95 rows x 2 columns]


## Data Cleaning

Clean the dataset: fill missing zip codes, drop duplicate rows, fix timeStamp to datetime type

In [8]:
# Check Missing values

df_calls.isnull().sum()

lat              0
lng              0
desc             0
zip          80199
title            0
timeStamp        0
twp            293
addr             0
e                0
dept             0
reason           0
dtype: int64

In [9]:
df_calls.shape

(663522, 11)

In [10]:
# Fill missing zip and twp with Unknown
df_calls['zip'] = df_calls['zip'].fillna('Unknown')
df_calls['twp'] = df_calls['twp'].fillna('Unknown')

In [11]:
df_calls.isnull().sum()

lat          0
lng          0
desc         0
zip          0
title        0
timeStamp    0
twp          0
addr         0
e            0
dept         0
reason       0
dtype: int64

In [12]:
df_calls.shape

(663522, 11)

In [13]:
# Check for duplicates
df_calls.duplicated().sum()

np.int64(240)

In [14]:
# Inspect duplicates
df_calls[df_calls.duplicated()]

,lat,lng,desc,zip,title,timeStamp,twp,addr,e,dept,reason
823,40.358269,-75.443173,UPPER RIDGE RD & PRICE RD; MARLBOROUGH; Stati...,18073.0,EMS: ALTERED MENTAL STATUS,2015-12-12 19:41:03,MARLBOROUGH,UPPER RIDGE RD & PRICE RD,1,EMS,ALTERED MENTAL STATUS
836,40.086010,-75.387142,DEKALB PIKE & CROCKETT RD; UPPER MERION; 2015-...,19406.0,Traffic: DISABLED VEHICLE -,2015-12-12 20:19:43,UPPER MERION,DEKALB PIKE & CROCKETT RD,1,Traffic,DISABLED VEHICLE -
845,40.133037,-75.408463,SHANNONDELL DR & SHANNONDELL BLVD; LOWER PROV...,19403.0,EMS: HEAD INJURY,2015-12-12 21:10:24,LOWER PROVIDENCE,SHANNONDELL DR & SHANNONDELL BLVD,1,EMS,HEAD INJURY
846,40.087320,-75.405055,N GULPH RD; UPPER MERION; 2015-12-12 @ 21:09:3...,Unknown,Fire: GAS-ODOR/LEAK,2015-12-12 21:09:39,UPPER MERION,N GULPH RD,1,Fire,GAS-ODOR/LEAK
847,40.087320,-75.405055,N GULPH RD; UPPER MERION; 2015-12-12 @ 21:09:3...,Unknown,Fire: GAS-ODOR/LEAK,2015-12-12 21:09:39,UPPER MERION,N GULPH RD,1,Fire,GAS-ODOR/LEAK
...,...,...,...,...,...,...,...,...,...,...,...
643583,40.149218,-75.318380,OAK TREE RD & PENN SQUARE RD; EAST NORRITON; 2...,19401.0,Fire: ELECTRICAL FIRE OUTSIDE,2020-06-03 20:36:12,EAST NORRITON,OAK TREE RD & PENN SQUARE RD,1,Fire,ELECTRICAL FIRE OUTSIDE
648409,40.245947,-75.250684,THOMAS RD & VILSMEIER RD; MONTGOMERY; Station...,19446.0,EMS: FALL VICTIM,2020-06-16 07:57:56,MONTGOMERY,THOMAS RD & VILSMEIER RD,1,EMS,FALL VICTIM
648410,40.245947,-75.250684,THOMAS RD & VILSMEIER RD; MONTGOMERY; Station...,19446.0,EMS: FALL VICTIM,2020-06-16 07:57:56,MONTGOMERY,THOMAS RD & VILSMEIER RD,1,EMS,FALL VICTIM
656197,40.021868,-75.317061,BRYN MAWR AVE; LOWER MERION; 2020-07-09 @ 08:2...,19010.0,Fire: TRANSFERRED CALL,2020-07-09 08:25:24,LOWER MERION,BRYN MAWR AVE,1,Fire,TRANSFERRED CALL


In [15]:
# Fix timestamp to datetime

df_calls['timeStamp'] = pd.to_datetime(df_calls['timeStamp'])

# Confirm the change
df_calls.info()

<class 'pandas.DataFrame'>
RangeIndex: 663522 entries, 0 to 663521
Data columns (total 11 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   lat        663522 non-null  float64       
 1   lng        663522 non-null  float64       
 2   desc       663522 non-null  str           
 3   zip        663522 non-null  object        
 4   title      663522 non-null  str           
 5   timeStamp  663522 non-null  datetime64[us]
 6   twp        663522 non-null  str           
 7   addr       663522 non-null  str           
 8   e          663522 non-null  int64         
 9   dept       663522 non-null  str           
 10  reason     663522 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(1), str(6)
memory usage: 152.3+ MB


## Features Engineering

In [16]:
# extract hour, day of week, and month from the timeStamp column
df_calls['hour'] = df_calls['timeStamp'].dt.hour
df_calls['day_of_week']= df_calls['timeStamp'].dt.day_name()
df_calls['month'] = df_calls['timeStamp'].dt.month

# Verify 
df_calls.info()

<class 'pandas.DataFrame'>
RangeIndex: 663522 entries, 0 to 663521
Data columns (total 14 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   lat          663522 non-null  float64       
 1   lng          663522 non-null  float64       
 2   desc         663522 non-null  str           
 3   zip          663522 non-null  object        
 4   title        663522 non-null  str           
 5   timeStamp    663522 non-null  datetime64[us]
 6   twp          663522 non-null  str           
 7   addr         663522 non-null  str           
 8   e            663522 non-null  int64         
 9   dept         663522 non-null  str           
 10  reason       663522 non-null  str           
 11  hour         663522 non-null  int32         
 12  day_of_week  663522 non-null  str           
 13  month        663522 non-null  int32         
dtypes: datetime64[us](1), float64(2), int32(2), int64(1), object(1), str(7)
memory usage: 167.0+ MB

## Data Preparation

### Encoding

In [17]:
# Apply One-Hot Encoding to the dept column using pd.get_dummies()
dummies = pd.get_dummies(df_calls.dept)
dummies = dummies.astype(int)
dummies

,EMS,Fire,Traffic
0,1,0,0
1,1,0,0
2,0,1,0
3,1,0,0
4,1,0,0
...,...,...,...
663517,0,0,1
663518,1,0,0
663519,1,0,0
663520,0,1,0


In [19]:
merged = pd.concat([df_calls, dummies], axis='columns')
merged

,lat,lng,desc,zip,title,timeStamp,twp,addr,e,dept,reason,hour,day_of_week,month,EMS,Fire,Traffic
0,40.297876,-75.581294,REINDEER CT & DEAD END; NEW HANOVER; Station ...,19525.0,EMS: BACK PAINS/INJURY,2015-12-10 17:10:52,NEW HANOVER,REINDEER CT & DEAD END,1,EMS,BACK PAINS/INJURY,17,Thursday,12,1,0,0
1,40.258061,-75.264680,BRIAR PATH & WHITEMARSH LN; HATFIELD TOWNSHIP...,19446.0,EMS: DIABETIC EMERGENCY,2015-12-10 17:29:21,HATFIELD TOWNSHIP,BRIAR PATH & WHITEMARSH LN,1,EMS,DIABETIC EMERGENCY,17,Thursday,12,1,0,0
2,40.121182,-75.351975,HAWS AVE; NORRISTOWN; 2015-12-10 @ 14:39:21-St...,19401.0,Fire: GAS-ODOR/LEAK,2015-12-10 14:39:21,NORRISTOWN,HAWS AVE,1,Fire,GAS-ODOR/LEAK,14,Thursday,12,0,1,0
3,40.116153,-75.343513,AIRY ST & SWEDE ST; NORRISTOWN; Station 308A;...,19401.0,EMS: CARDIAC EMERGENCY,2015-12-10 16:47:36,NORRISTOWN,AIRY ST & SWEDE ST,1,EMS,CARDIAC EMERGENCY,16,Thursday,12,1,0,0
4,40.251492,-75.603350,CHERRYWOOD CT & DEAD END; LOWER POTTSGROVE; S...,Unknown,EMS: DIZZINESS,2015-12-10 16:56:52,LOWER POTTSGROVE,CHERRYWOOD CT & DEAD END,1,EMS,DIZZINESS,16,Thursday,12,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
663517,40.157956,-75.348060,SUNSET AVE & WOODLAND AVE; EAST NORRITON; 2020...,19403.0,Traffic: VEHICLE ACCIDENT -,2020-07-29 15:46:51,EAST NORRITON,SUNSET AVE & WOODLAND AVE,1,Traffic,VEHICLE ACCIDENT -,15,Wednesday,7,0,0,1
663518,40.136306,-75.428697,EAGLEVILLE RD & BUNTING CIR; LOWER PROVIDENCE...,19403.0,EMS: GENERAL WEAKNESS,2020-07-29 15:52:19,LOWER PROVIDENCE,EAGLEVILLE RD & BUNTING CIR,1,EMS,GENERAL WEAKNESS,15,Wednesday,7,1,0,0
663519,40.013779,-75.300835,HAVERFORD STATION RD; LOWER MERION; Station 3...,19041.0,EMS: VEHICLE ACCIDENT,2020-07-29 15:52:52,LOWER MERION,HAVERFORD STATION RD,1,EMS,VEHICLE ACCIDENT,15,Wednesday,7,1,0,0
663520,40.121603,-75.351437,MARSHALL ST & HAWS AVE; NORRISTOWN; 2020-07-29...,19401.0,Fire: BUILDING FIRE,2020-07-29 15:54:08,NORRISTOWN,MARSHALL ST & HAWS AVE,1,Fire,BUILDING FIRE,15,Wednesday,7,0,1,0


In [20]:

final_frame = merged.drop(['dept'], axis='columns')
final_frame

,lat,lng,desc,zip,title,timeStamp,twp,addr,e,reason,hour,day_of_week,month,EMS,Fire,Traffic
0,40.297876,-75.581294,REINDEER CT & DEAD END; NEW HANOVER; Station ...,19525.0,EMS: BACK PAINS/INJURY,2015-12-10 17:10:52,NEW HANOVER,REINDEER CT & DEAD END,1,BACK PAINS/INJURY,17,Thursday,12,1,0,0
1,40.258061,-75.264680,BRIAR PATH & WHITEMARSH LN; HATFIELD TOWNSHIP...,19446.0,EMS: DIABETIC EMERGENCY,2015-12-10 17:29:21,HATFIELD TOWNSHIP,BRIAR PATH & WHITEMARSH LN,1,DIABETIC EMERGENCY,17,Thursday,12,1,0,0
2,40.121182,-75.351975,HAWS AVE; NORRISTOWN; 2015-12-10 @ 14:39:21-St...,19401.0,Fire: GAS-ODOR/LEAK,2015-12-10 14:39:21,NORRISTOWN,HAWS AVE,1,GAS-ODOR/LEAK,14,Thursday,12,0,1,0
3,40.116153,-75.343513,AIRY ST & SWEDE ST; NORRISTOWN; Station 308A;...,19401.0,EMS: CARDIAC EMERGENCY,2015-12-10 16:47:36,NORRISTOWN,AIRY ST & SWEDE ST,1,CARDIAC EMERGENCY,16,Thursday,12,1,0,0
4,40.251492,-75.603350,CHERRYWOOD CT & DEAD END; LOWER POTTSGROVE; S...,Unknown,EMS: DIZZINESS,2015-12-10 16:56:52,LOWER POTTSGROVE,CHERRYWOOD CT & DEAD END,1,DIZZINESS,16,Thursday,12,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
663517,40.157956,-75.348060,SUNSET AVE & WOODLAND AVE; EAST NORRITON; 2020...,19403.0,Traffic: VEHICLE ACCIDENT -,2020-07-29 15:46:51,EAST NORRITON,SUNSET AVE & WOODLAND AVE,1,VEHICLE ACCIDENT -,15,Wednesday,7,0,0,1
663518,40.136306,-75.428697,EAGLEVILLE RD & BUNTING CIR; LOWER PROVIDENCE...,19403.0,EMS: GENERAL WEAKNESS,2020-07-29 15:52:19,LOWER PROVIDENCE,EAGLEVILLE RD & BUNTING CIR,1,GENERAL WEAKNESS,15,Wednesday,7,1,0,0
663519,40.013779,-75.300835,HAVERFORD STATION RD; LOWER MERION; Station 3...,19041.0,EMS: VEHICLE ACCIDENT,2020-07-29 15:52:52,LOWER MERION,HAVERFORD STATION RD,1,VEHICLE ACCIDENT,15,Wednesday,7,1,0,0
663520,40.121603,-75.351437,MARSHALL ST & HAWS AVE; NORRISTOWN; 2020-07-29...,19401.0,Fire: BUILDING FIRE,2020-07-29 15:54:08,NORRISTOWN,MARSHALL ST & HAWS AVE,1,BUILDING FIRE,15,Wednesday,7,0,1,0


In [21]:
final_frame.info()

<class 'pandas.DataFrame'>
RangeIndex: 663522 entries, 0 to 663521
Data columns (total 16 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   lat          663522 non-null  float64       
 1   lng          663522 non-null  float64       
 2   desc         663522 non-null  str           
 3   zip          663522 non-null  object        
 4   title        663522 non-null  str           
 5   timeStamp    663522 non-null  datetime64[us]
 6   twp          663522 non-null  str           
 7   addr         663522 non-null  str           
 8   e            663522 non-null  int64         
 9   reason       663522 non-null  str           
 10  hour         663522 non-null  int32         
 11  day_of_week  663522 non-null  str           
 12  month        663522 non-null  int32         
 13  EMS          663522 non-null  int64         
 14  Fire         663522 non-null  int64         
 15  Traffic      663522 non-null  int64         


In [22]:
final_frame.nunique()

lat             25949
lng             25980
desc           663282
zip               205
title             148
timeStamp      640754
twp                69
addr            41292
e                   1
reason             95
hour               24
day_of_week         7
month              12
EMS                 2
Fire                2
Traffic             2
dtype: int64

In [196]:
cols_to_drop = ['zip', 'title', 'addr', 'twp', 'timeStamp', 'desc', 'e']

final_frame = final_frame.drop(cols_to_drop, axis=1)
final_frame.head()

,lat,lng,reason,hour,day_of_week,month,EMS,Fire,Traffic
0,40.297876,-75.581294,BACK PAINS/INJURY,17,Thursday,12,1,0,0
1,40.258061,-75.264680,DIABETIC EMERGENCY,17,Thursday,12,1,0,0
2,40.121182,-75.351975,GAS-ODOR/LEAK,14,Thursday,12,0,1,0
3,40.116153,-75.343513,CARDIAC EMERGENCY,16,Thursday,12,1,0,0
4,40.251492,-75.603350,DIZZINESS,16,Thursday,12,1,0,0


In [23]:
# Encode days of the week
final_frame = pd.get_dummies(
    final_frame,
    columns=['day_of_week'],
    dtype=int
)

# Copy dataframe to the new one
df = final_frame.copy()
df.head()

,lat,lng,desc,zip,title,timeStamp,twp,addr,e,reason,...,EMS,Fire,Traffic,day_of_week_Friday,day_of_week_Monday,day_of_week_Saturday,day_of_week_Sunday,day_of_week_Thursday,day_of_week_Tuesday,day_of_week_Wednesday
0,40.297876,-75.581294,REINDEER CT & DEAD END; NEW HANOVER; Station ...,19525.0,EMS: BACK PAINS/INJURY,2015-12-10 17:10:52,NEW HANOVER,REINDEER CT & DEAD END,1,BACK PAINS/INJURY,...,1,0,0,0,0,0,0,1,0,0
1,40.258061,-75.264680,BRIAR PATH & WHITEMARSH LN; HATFIELD TOWNSHIP...,19446.0,EMS: DIABETIC EMERGENCY,2015-12-10 17:29:21,HATFIELD TOWNSHIP,BRIAR PATH & WHITEMARSH LN,1,DIABETIC EMERGENCY,...,1,0,0,0,0,0,0,1,0,0
2,40.121182,-75.351975,HAWS AVE; NORRISTOWN; 2015-12-10 @ 14:39:21-St...,19401.0,Fire: GAS-ODOR/LEAK,2015-12-10 14:39:21,NORRISTOWN,HAWS AVE,1,GAS-ODOR/LEAK,...,0,1,0,0,0,0,0,1,0,0
3,40.116153,-75.343513,AIRY ST & SWEDE ST; NORRISTOWN; Station 308A;...,19401.0,EMS: CARDIAC EMERGENCY,2015-12-10 16:47:36,NORRISTOWN,AIRY ST & SWEDE ST,1,CARDIAC EMERGENCY,...,1,0,0,0,0,0,0,1,0,0
4,40.251492,-75.603350,CHERRYWOOD CT & DEAD END; LOWER POTTSGROVE; S...,Unknown,EMS: DIZZINESS,2015-12-10 16:56:52,LOWER POTTSGROVE,CHERRYWOOD CT & DEAD END,1,DIZZINESS,...,1,0,0,0,0,0,0,1,0,0


In [199]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 663522 entries, 0 to 663521
Data columns (total 15 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   lat                    663522 non-null  float64
 1   lng                    663522 non-null  float64
 2   reason                 663522 non-null  str    
 3   hour                   663522 non-null  int32  
 4   month                  663522 non-null  int32  
 5   EMS                    663522 non-null  int64  
 6   Fire                   663522 non-null  int64  
 7   Traffic                663522 non-null  int64  
 8   day_of_week_Friday     663522 non-null  int64  
 9   day_of_week_Monday     663522 non-null  int64  
 10  day_of_week_Saturday   663522 non-null  int64  
 11  day_of_week_Sunday     663522 non-null  int64  
 12  day_of_week_Thursday   663522 non-null  int64  
 13  day_of_week_Tuesday    663522 non-null  int64  
 14  day_of_week_Wednesday  663522 non-null  int64  

### Data Splitting

In [24]:
# Separate features from target
X = df.drop(columns=['reason'])
y = df['reason']

X.shape, y.shape

((663522, 21), (663522,))

In [25]:
# Split dataset into train and test sets

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [202]:
print(len(X_train), len(X_test))
print(len(y_train), len(y_test))

530817 132705
530817 132705


In [27]:
# Scale Features lat and lng using MInMaxScaler

scaler = MinMaxScaler()
X_train[['lat', 'lng']] = scaler.fit_transform(X_train[['lat', 'lng']])
X_test[['lat', 'lng']] = scaler.transform(X_test[['lat', 'lng']])


In [36]:
# Confirm the shape of X_train and X_test

X_train.shape, X_test.shape

((530817, 21), (132705, 21))